# Week 5 Day 1 
 Raw Python Agent

In [17]:
import os
import json
from typing import Any, Dict, List

from groq import Groq
from dotenv import load_dotenv

load_dotenv()

MODEL = "llama-3.3-70b-versatile"
MAX_ITERATIONS = 10

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

### Tool Implementations

In [18]:
def calculator(a: float, b: float, operation: str) -> Dict[str, Any]:
    """Perform basic arithmetic."""
    try:
        if operation == "add":
            result = a + b
        elif operation == "subtract":
            result = a - b
        elif operation == "multiply":
            result = a * b
        elif operation == "divide":
            if b == 0:
                return {"success": False, "error": "Cannot divide by zero."}
            result = a / b
        else:
            return {"success": False, "error": f"Unknown operation: {operation}"}
        return {"success": True, "result": result}
    except Exception as e:
        return {"success": False, "error": str(e)}

In [19]:
WEATHER_DATA = {
    "lahore": {"temperature": 32, "condition": "Sunny"},
    "islamabad": {"temperature": 28, "condition": "Partly cloudy"},
    "karachi": {"temperature": 31, "condition": "Humid"},
    "faisalabad": {"temperature": 33, "condition": "Sunny"},
}

def weather_lookup(city: str) -> Dict[str, Any]:
    """Look up weather information for a city."""
    city_key = city.strip().lower()
    if city_key not in WEATHER_DATA:
        return {"success": False, "error": f"Weather data not available for '{city}'."}
    weather = WEATHER_DATA[city_key]
    return {
        "success": True,
        "city": city,
        "temperature": weather["temperature"],
        "condition": weather["condition"],
    }

### Tool Schemas 
(OpenAI-compatible format Groq expects)

In [20]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": (
                "Perform basic arithmetic calculations. Use this tool when "
                "the user asks for addition, subtraction, multiplication, "
                "or division of two numbers."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "The first number."},
                    "b": {"type": "number", "description": "The second number."},
                    "operation": {
                        "type": "string",
                        "description": "The arithmetic operation to perform.",
                        "enum": ["add", "subtract", "multiply", "divide"],
                    },
                },
                "required": ["a", "b", "operation"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "weather_lookup",
            "description": (
                "Look up current weather information for a named city. Use "
                "this tool when the user asks about weather, temperature, "
                "or conditions in a specific place. Only works for cities "
                "in the local dataset (Lahore, Islamabad, Karachi, "
                "Faisalabad)."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The city name."}
                },
                "required": ["city"],
            },
        },
    },
]

TOOL_REGISTRY = {
    "calculator": calculator,
    "weather_lookup": weather_lookup,
}

### Tool Execution

In [21]:
def execute_tool(tool_name: str, tool_input: Dict[str, Any]) -> Dict[str, Any]:
    """Execute the requested Python tool, with guardrails."""
    print("\n" + "=" * 60)
    print("TOOL CALL")
    print("=" * 60)
    print(f"Tool: {tool_name}")
    print("Arguments:", json.dumps(tool_input, indent=2))

    if tool_name not in TOOL_REGISTRY:
        error = {"success": False, "error": f"Unknown tool: {tool_name}"}
        print("ERROR:", error)
        return error

    tool_function = TOOL_REGISTRY[tool_name]

    try:
        result = tool_function(**tool_input)
        print("\nTOOL RESULT:")
        print(json.dumps(result, indent=2))
        return result
    except TypeError as e:
        error = {"success": False, "error": f"Invalid tool arguments: {str(e)}"}
        print("ERROR:", error)
        return error
    except Exception as e:
        error = {"success": False, "error": f"Tool execution failed: {str(e)}"}
        print("ERROR:", error)
        return error

In [22]:
def safe_api_call(messages: List[Dict[str, Any]]):
    """Wrap the Groq call so network/rate-limit errors don't crash the loop."""
    try:
        return client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0,
        )
    except Exception as e:
        print(f"\nAPI ERROR: {e}")
        return None

### Task 2 
 Single Tool Call Demo

In [23]:
def single_tool_call_demo():
    print("\n" + "#" * 70)
    print("TASK 2 -- SINGLE TOOL CALL")
    print("#" * 70)

    messages = [{"role": "user", "content": "Calculate 25 multiplied by 17."}]

    response = safe_api_call(messages)
    if response is None:
        print("Aborting demo: API call failed.")
        return

    message = response.choices[0].message
    print("MODEL RESPONSE:")
    if message.content:
        print("  [reasoning]", message.content)
    if message.tool_calls:
        for tc in message.tool_calls:
            print(f"  [tool_call] {tc.function.name}({tc.function.arguments})")

    if not message.tool_calls:
        print("\nThe model did not request a tool.")
        return

    messages.append(message)
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        tool_input = json.loads(tool_call.function.arguments)
        result = execute_tool(tool_name, tool_input)
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result),
            }
        )

    final_response = safe_api_call(messages)
    if final_response is None:
        print("Aborting demo: follow-up API call failed.")
        return

    final_message = final_response.choices[0].message
    print("\nFINAL ANSWER:")
    print(final_message.content)

###  Task 4 Memory and Working State
Separates **conversation memory** (`self.messages`  full history sent back to the model) from **working memory** (`self.state`  the agent's own scratchpad for logging, debugging, and guardrails).

In [24]:
class AgentState:
    def __init__(self):
        self.messages: List[Dict[str, Any]] = []
        self.state = {
            "iterations": 0,
            "tool_calls": 0,
            "tools_used": [],
            "observations": [],
            "reasoning_log": [],
        }

    def add_tool_usage(self, tool_name: str, tool_input: Dict[str, Any], observation: Dict[str, Any]):
        self.state["tool_calls"] += 1
        self.state["tools_used"].append(tool_name)
        self.state["observations"].append(
            {"tool": tool_name, "input": tool_input, "output": observation}
        )

    def add_reasoning(self, iteration: int, text: str):
        if text and text.strip():
            self.state["reasoning_log"].append({"iteration": iteration, "text": text})

### Task 3
Raw Python Agent Loop (ReAct) Includes a guardrail that stops the loop if the model repeats the exact same tool call three times in a row (a common infinite-loop failure mode), in addition to the `MAX_ITERATIONS` cap.

In [25]:
def run_agent(user_request: str) -> str:
    agent_state = AgentState()
    agent_state.messages.append({"role": "user", "content": user_request})

    print("\n" + "#" * 70)
    print("AGENT STARTED")
    print("#" * 70)
    print("\nUSER REQUEST:")
    print(user_request)

    recent_calls: List[str] = []

    for iteration in range(1, MAX_ITERATIONS + 1):
        agent_state.state["iterations"] = iteration
        print("\n" + "-" * 70)
        print(f"ITERATION {iteration}")
        print("-" * 70)

        response = safe_api_call(agent_state.messages)
        if response is None:
            return "The agent stopped because the API call failed. See logs above."

        message = response.choices[0].message

        if message.content:
            print("\nMODEL REASONING:")
            print(message.content)
            agent_state.add_reasoning(iteration, message.content)

        if not message.tool_calls:
            final_text = message.content or ""
            print("\nFINAL RESPONSE:")
            print(final_text)
            print("\nFINAL AGENT STATE:")
            print(json.dumps(agent_state.state, indent=2))
            return final_text

        print("\nMODEL REQUESTED TOOL(S):")
        for tc in message.tool_calls:
            print(f"Tool: {tc.function.name}  Arguments: {tc.function.arguments}")

        agent_state.messages.append(message)

        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name

            try:
                tool_input = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError as e:
                result = {"success": False, "error": f"Invalid JSON arguments: {str(e)}"}
                print("\nARGUMENT PARSING ERROR:")
                print(result)
                agent_state.add_tool_usage(tool_name, {}, result)
                agent_state.messages.append(
                    {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
                )
                continue

            call_signature = f"{tool_name}:{json.dumps(tool_input, sort_keys=True)}"
            recent_calls.append(call_signature)
            if len(recent_calls) >= 3 and len(set(recent_calls[-3:])) == 1:
                print("\nGUARDRAIL TRIGGERED: identical tool call repeated 3x in a row.")
                result = {
                    "success": False,
                    "error": (
                        "Guardrail: this exact tool call has been repeated "
                        "three times in a row. Stopping to prevent an "
                        "infinite loop."
                    ),
                }
            else:
                result = execute_tool(tool_name, tool_input)

            agent_state.add_tool_usage(tool_name, tool_input, result)
            agent_state.messages.append(
                {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
            )

        print("\nOBSERVATION RETURNED TO MODEL.")

    print("\n" + "#" * 70)
    print("MAX ITERATIONS REACHED")
    print("#" * 70)
    return f"The agent stopped because it reached the maximum iteration limit of {MAX_ITERATIONS}."

### Task 5
Failure Mode Tests

In [26]:
def test_tool_error():
    print("\n" + "#" * 70)
    print("FAILURE TEST 1 -- TOOL ERROR (unsupported city)")
    print("#" * 70)
    result = execute_tool("weather_lookup", {"city": "Atlantis"})
    print("\nOBSERVED BEHAVIOR:")
    print(json.dumps(result, indent=2))


def test_unknown_tool():
    print("\n" + "#" * 70)
    print("FAILURE TEST 2 -- UNKNOWN / HALLUCINATED TOOL")
    print("#" * 70)
    result = execute_tool("send_email", {"recipient": "test@example.com"})
    print("\nOBSERVED BEHAVIOR:")
    print(json.dumps(result, indent=2))


def test_invalid_arguments():
    print("\n" + "#" * 70)
    print("FAILURE TEST 3 -- INVALID ARGUMENTS")
    print("#" * 70)
    result = execute_tool("calculator", {"a": 10, "b": 5, "operation": "square_root"})
    print("\nOBSERVED BEHAVIOR:")
    print(json.dumps(result, indent=2))


def test_ambiguous_request():
    print("\n" + "#" * 70)
    print("FAILURE TEST 4 -- AMBIGUOUS REQUEST")
    print("#" * 70)
    result = run_agent("What's the weather like there?")
    print("\nOBSERVED BEHAVIOR (final agent output):")
    print(result)


def test_missing_tool_capability():
    print("\n" + "#" * 70)
    print("FAILURE TEST 5 -- REQUEST REQUIRES AN UNDEFINED TOOL")
    print("#" * 70)
    result = run_agent(
        "Convert the temperature in Karachi from Celsius to Fahrenheit "
        "and email it to my manager."
    )
    print("\nOBSERVED BEHAVIOR (final agent output):")
    print(result)


def test_loop_risk():
    print("\n" + "#" * 70)
    print("FAILURE TEST 6 -- REPEATED IDENTICAL TOOL CALL / LOOP RISK")
    print("#" * 70)
    result = run_agent(
        "Keep checking the weather in Lahore over and over until it stops "
        "being sunny."
    )
    print("\nOBSERVED BEHAVIOR (final agent output):")
    print(result)

 8. Run Everything\n\nMake sure `GROQ_API_KEY` is set (via `.env`) before running this cell.

In [27]:
if not os.environ.get("GROQ_API_KEY"):
    print("ERROR: GROQ_API_KEY is not set.")
    print("Set it in a .env file or your environment before running.")
else:
    single_tool_call_demo()


######################################################################
TASK 2 -- SINGLE TOOL CALL
######################################################################
MODEL RESPONSE:
  [tool_call] calculator({"a":25,"b":17,"operation":"multiply"})

TOOL CALL
Tool: calculator
Arguments: {
  "a": 25,
  "b": 17,
  "operation": "multiply"
}

TOOL RESULT:
{
  "success": true,
  "result": 425
}

FINAL ANSWER:
The result of 25 multiplied by 17 is 425.


In [28]:
run_agent(
    "Look up the weather in Lahore and Islamabad. "
    "Tell me which city is warmer and by how many degrees."
)


######################################################################
AGENT STARTED
######################################################################

USER REQUEST:
Look up the weather in Lahore and Islamabad. Tell me which city is warmer and by how many degrees.

----------------------------------------------------------------------
ITERATION 1
----------------------------------------------------------------------

MODEL REQUESTED TOOL(S):
Tool: weather_lookup  Arguments: {"city":"Lahore"}
Tool: weather_lookup  Arguments: {"city":"Islamabad"}

TOOL CALL
Tool: weather_lookup
Arguments: {
  "city": "Lahore"
}

TOOL RESULT:
{
  "success": true,
  "city": "Lahore",
  "temperature": 32,
  "condition": "Sunny"
}

TOOL CALL
Tool: weather_lookup
Arguments: {
  "city": "Islamabad"
}

TOOL RESULT:
{
  "success": true,
  "city": "Islamabad",
  "temperature": 28,
  "condition": "Partly cloudy"
}

OBSERVATION RETURNED TO MODEL.

--------------------------------------------------------------

'Lahore is warmer than Islamabad by 4 degrees.'

In [29]:
test_tool_error()
test_unknown_tool()
test_invalid_arguments()


######################################################################
FAILURE TEST 1 -- TOOL ERROR (unsupported city)
######################################################################

TOOL CALL
Tool: weather_lookup
Arguments: {
  "city": "Atlantis"
}

TOOL RESULT:
{
  "success": false,
  "error": "Weather data not available for 'Atlantis'."
}

OBSERVED BEHAVIOR:
{
  "success": false,
  "error": "Weather data not available for 'Atlantis'."
}

######################################################################
FAILURE TEST 2 -- UNKNOWN / HALLUCINATED TOOL
######################################################################

TOOL CALL
Tool: send_email
Arguments: {
  "recipient": "test@example.com"
}
ERROR: {'success': False, 'error': 'Unknown tool: send_email'}

OBSERVED BEHAVIOR:
{
  "success": false,
  "error": "Unknown tool: send_email"
}

######################################################################
FAILURE TEST 3 -- INVALID ARGUMENTS
############################

In [30]:
test_ambiguous_request()


######################################################################
FAILURE TEST 4 -- AMBIGUOUS REQUEST
######################################################################

######################################################################
AGENT STARTED
######################################################################

USER REQUEST:
What's the weather like there?

----------------------------------------------------------------------
ITERATION 1
----------------------------------------------------------------------

MODEL REQUESTED TOOL(S):
Tool: weather_lookup  Arguments: {"city":"Lahore"}

TOOL CALL
Tool: weather_lookup
Arguments: {
  "city": "Lahore"
}

TOOL RESULT:
{
  "success": true,
  "city": "Lahore",
  "temperature": 32,
  "condition": "Sunny"
}

OBSERVATION RETURNED TO MODEL.

----------------------------------------------------------------------
ITERATION 2
----------------------------------------------------------------------

MODEL REQUESTED TOOL(S):
Tool: 

In [31]:
test_missing_tool_capability()


######################################################################
FAILURE TEST 5 -- REQUEST REQUIRES AN UNDEFINED TOOL
######################################################################

######################################################################
AGENT STARTED
######################################################################

USER REQUEST:
Convert the temperature in Karachi from Celsius to Fahrenheit and email it to my manager.

----------------------------------------------------------------------
ITERATION 1
----------------------------------------------------------------------

MODEL REQUESTED TOOL(S):
Tool: weather_lookup  Arguments: {"city":"Karachi"}

TOOL CALL
Tool: weather_lookup
Arguments: {
  "city": "Karachi"
}

TOOL RESULT:
{
  "success": true,
  "city": "Karachi",
  "temperature": 31,
  "condition": "Humid"
}

OBSERVATION RETURNED TO MODEL.

----------------------------------------------------------------------
ITERATION 2
------------------------

In [32]:
test_loop_risk()


######################################################################
FAILURE TEST 6 -- REPEATED IDENTICAL TOOL CALL / LOOP RISK
######################################################################

######################################################################
AGENT STARTED
######################################################################

USER REQUEST:
Keep checking the weather in Lahore over and over until it stops being sunny.

----------------------------------------------------------------------
ITERATION 1
----------------------------------------------------------------------

MODEL REQUESTED TOOL(S):
Tool: weather_lookup  Arguments: {"city":"Lahore"}
Tool: weather_lookup  Arguments: {"city":"Lahore"}
Tool: weather_lookup  Arguments: {"city":"Lahore"}

TOOL CALL
Tool: weather_lookup
Arguments: {
  "city": "Lahore"
}

TOOL RESULT:
{
  "success": true,
  "city": "Lahore",
  "temperature": 32,
  "condition": "Sunny"
}

TOOL CALL
Tool: weather_lookup
Arguments: {
  